# DNA-aware stochastic model with mapping rate + surrogate search (AUTO-RUN)

This notebook implements the pipeline:

DNA inputs (HDR ng, sgRNA ng, ratio)
→ predicted HDR rate (Hill model)
→ stochastic simulator (library skew + sampling)
→ distributions of outcomes (dropout, p10 reads)
→ synthetic training data (simulated)
→ surrogate models (fast)
→ inverse design (surrogate search + Monte Carlo verification)

**New in this version:** mapping rate is explicitly modeled (typical 50–60%).
Usable reads are `reads_usable = reads_total × mapping_rate`.


In [1]:
import os, sys
sys.path.append('.')
sys.path.append('/Users/ds39/PycharmProjects/MAVE_Project/Simulation_Prediction_modelling/26_feb_modelling')
sys.path.append('/mnt/data')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sge_model_skew_dna_mapping_v3 as mod
print('Imported:', mod.__file__)


Imported: /Users/ds39/PycharmProjects/MAVE_Project/Simulation_Prediction_modelling/26_feb_modelling/sge_model_skew_dna_mapping_v3.py


In [2]:
# -----------------------------
# SETTINGS (edit these)
# -----------------------------
OUTDIR = '.'  # e.g. './outputs'
os.makedirs(OUTDIR, exist_ok=True)

# DNA inputs for a single example run
HDR_NG_EXAMPLE = 750
SGRNA_NG_EXAMPLE = 375
RATIO_OPT = 2.0

# Experimental + sequencing settings
CELLS_TRANSFECTED = 200_000
LIBRARY_SIZE = 2000
READS_TOTAL = 1_000_000

# Library skew
SKEW_SIGMA = 0.5

# Mapping rate settings
# Option 1 (fixed): set MAPPING_RATE_FIXED to a number, e.g. 0.55
# Option 2 (variable): set MAPPING_RATE_FIXED = None and choose mean + kappa
MAPPING_RATE_FIXED = 0.55
MAPPING_MEAN = 0.55
MAPPING_KAPPA = 60.0

# Monte Carlo reps
N_REPS_EXAMPLE = 50

# Grid sweep (for plots)
HDR_VALUES = np.linspace(100, 2000, 8)
SGRNA_VALUES = np.linspace(50, 1500, 8)
N_REPS_GRID = 25

# Synthetic training data
N_SYNTH = 500
N_REPS_PER_SYNTH = 12
MAPPING_RANGE_SYNTH = (0.40, 0.70)

# Surrogate + verification (Option B)
N_CANDIDATES = 20000
TOP_K_VERIFY = 60
N_REPS_VERIFY = 40

# Targets (edit these)
TARGET_DROPOUT = 0.02
MIN_P10_READS = 100

print('OUTDIR =', OUTDIR)


OUTDIR = .


In [3]:
# -----------------------------
# 1) Example run: report predicted HDR rate + mapping rate + usable reads
# -----------------------------
mc = mod.run_monte_carlo(
    n_reps=N_REPS_EXAMPLE,
    hdr_ng=HDR_NG_EXAMPLE,
    sgrna_ng=SGRNA_NG_EXAMPLE,
    ratio_opt=RATIO_OPT,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    skew_sigma=SKEW_SIGMA,
    reads_total=READS_TOTAL,
    mapping_rate=MAPPING_RATE_FIXED,
    mapping_mean=MAPPING_MEAN,
    mapping_kappa=MAPPING_KAPPA,
)

print('Predicted HDR rate (mean):', mc['hdr_rate_mean'])
print('Assumed/predicted mapping rate (mean):', mc['mapping_rate_mean'])
print('Expected usable reads (mean):', mc['reads_usable_mean'])
print('Dropout mean:', mc['dropout_mean'], 'q05..q95:', mc['dropout_q']['0.05'], mc['dropout_q']['0.95'])
print('P10 reads mean:', mc['p10_reads_mean'], 'q05..q95:', mc['p10_reads_q']['0.05'], mc['p10_reads_q']['0.95'])


Predicted HDR rate (mean): 0.2590619136960601
Assumed/predicted mapping rate (mean): 0.55
Expected usable reads (mean): 550000.0
Dropout mean: 3e-05 q05..q95: 0.0 0.00027499999999999855
P10 reads mean: 113.318 q05..q95: 109.0 117.0


In [4]:
# -----------------------------
# 2) Coverage distribution example (histogram + CDF) for ONE replicate
# -----------------------------
rep = mod.simulate_once(
    hdr_ng=HDR_NG_EXAMPLE,
    sgrna_ng=SGRNA_NG_EXAMPLE,
    ratio_opt=RATIO_OPT,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    skew_sigma=SKEW_SIGMA,
    reads_total=READS_TOTAL,
    mapping_rate=MAPPING_RATE_FIXED,
    mapping_mean=MAPPING_MEAN,
    mapping_kappa=MAPPING_KAPPA,
    return_vectors=True,
)

read_counts = rep['read_counts']

# Histogram
plt.figure()
plt.hist(read_counts, bins=50)
plt.xlabel('Reads per variant')
plt.ylabel('Count of variants')
plt.title('Read-count distribution (one replicate)')
plt.tight_layout()
hist_path = os.path.join(OUTDIR, 'readcount_histogram.png')
plt.savefig(hist_path, dpi=200)
plt.close()

# CDF
sorted_counts = np.sort(read_counts)
cdf = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)
plt.figure()
plt.plot(sorted_counts, cdf)
plt.xlabel('Reads per variant')
plt.ylabel('CDF')
plt.title('CDF of reads per variant (one replicate)')
plt.grid(True)
plt.tight_layout()
cdf_path = os.path.join(OUTDIR, 'readcount_cdf.png')
plt.savefig(cdf_path, dpi=200)
plt.close()

print('Saved distribution plots:', hist_path, cdf_path)


Saved distribution plots: ./readcount_histogram.png ./readcount_cdf.png


In [5]:
# -----------------------------
# 3) Grid sweep (HDR x sgRNA) with mapping rate applied
# -----------------------------
dropout_grid = np.zeros((len(HDR_VALUES), len(SGRNA_VALUES)))
p10_grid = np.zeros_like(dropout_grid)
hdr_rate_grid = np.zeros_like(dropout_grid)

for i, h in enumerate(HDR_VALUES):
    for j, s in enumerate(SGRNA_VALUES):
        mc = mod.run_monte_carlo(
            n_reps=N_REPS_GRID,
            hdr_ng=float(h),
            sgrna_ng=float(s),
            ratio_opt=RATIO_OPT,
            cells_transfected=CELLS_TRANSFECTED,
            library_size=LIBRARY_SIZE,
            skew_sigma=SKEW_SIGMA,
            reads_total=READS_TOTAL,
            mapping_rate=MAPPING_RATE_FIXED,
            mapping_mean=MAPPING_MEAN,
            mapping_kappa=MAPPING_KAPPA,
        )
        dropout_grid[i, j] = mc['dropout_mean']
        p10_grid[i, j] = mc['p10_reads_mean']
        hdr_rate_grid[i, j] = mc['hdr_rate_mean']

print('Sweep done. Dropout range:', dropout_grid.min(), 'to', dropout_grid.max())


Sweep done. Dropout range: 0.0 to 0.41336


In [6]:
# -----------------------------
# 4) Save sweep plots
# -----------------------------
dropout_vs_hdr = dropout_grid.mean(axis=1)
p10_vs_hdr = p10_grid.mean(axis=1)

plt.figure()
plt.plot(HDR_VALUES, dropout_vs_hdr)
plt.xlabel('HDR (ng)')
plt.ylabel('Mean dropout')
plt.title('Mean dropout vs HDR (sgRNA averaged)')
plt.grid(True)
plt.tight_layout()
plot_dropout_vs_hdr = os.path.join(OUTDIR, 'plot_dropout_vs_hdr.png')
plt.savefig(plot_dropout_vs_hdr, dpi=200)
plt.close()

plt.figure()
plt.plot(HDR_VALUES, p10_vs_hdr)
plt.xlabel('HDR (ng)')
plt.ylabel('Mean p10 reads')
plt.title('Mean p10 reads vs HDR (sgRNA averaged)')
plt.grid(True)
plt.tight_layout()
plot_p10_vs_hdr = os.path.join(OUTDIR, 'plot_p10_vs_hdr.png')
plt.savefig(plot_p10_vs_hdr, dpi=200)
plt.close()

plt.figure()
plt.imshow(dropout_grid, aspect='auto', origin='lower')
plt.xticks(ticks=np.arange(len(SGRNA_VALUES)), labels=[f'{v:.0f}' for v in SGRNA_VALUES], rotation=45)
plt.yticks(ticks=np.arange(len(HDR_VALUES)), labels=[f'{v:.0f}' for v in HDR_VALUES])
plt.xlabel('sgRNA (ng)')
plt.ylabel('HDR (ng)')
plt.title('Dropout heatmap (HDR x sgRNA)')
plt.colorbar(label='Mean dropout')
plt.tight_layout()
dropout_heatmap = os.path.join(OUTDIR, 'dropout_heatmap.png')
plt.savefig(dropout_heatmap, dpi=200)
plt.close()

print('Saved sweep plots:', plot_dropout_vs_hdr, plot_p10_vs_hdr, dropout_heatmap)


Saved sweep plots: ./plot_dropout_vs_hdr.png ./plot_p10_vs_hdr.png ./dropout_heatmap.png


In [7]:
# -----------------------------
# 5) Build synthetic training data and SAVE it
# -----------------------------
rows = mod.build_synthetic_dataset(
    n_samples=N_SYNTH,
    n_reps_per_sample=N_REPS_PER_SYNTH,
    reads_total=READS_TOTAL,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    ratio_opt=RATIO_OPT,
    mapping_range=MAPPING_RANGE_SYNTH,
)
df_train = pd.DataFrame(rows)
train_csv = os.path.join(OUTDIR, 'synthetic_training_data.csv')
df_train.to_csv(train_csv, index=False)
print('Saved synthetic training data:', train_csv)
df_train.head(10)


Saved synthetic training data: ./synthetic_training_data.csv


,HDR_ng,sgRNA_ng,ratio,skew_sigma,mapping_rate,reads_total,cells_transfected,library_size,dropout_mean,p10_reads_mean,hdr_rate_mean,reads_usable_mean
0,981.027964,132.870157,7.383358,0.675767,0.474689,1000000,200000,2000,0.446542,0.000000,0.010000,474689.0
1,481.059102,89.430644,5.379130,0.181667,0.518664,1000000,200000,2000,0.370375,0.000000,0.010077,518664.0
2,1326.217401,395.878447,3.350062,0.388929,0.630793,1000000,200000,2000,0.000292,144.908333,0.141906,630793.0
3,214.736552,1466.863113,0.146392,1.275228,0.513791,1000000,200000,2000,0.362500,0.000000,0.025824,513790.0
4,204.655600,613.852867,0.333395,0.804811,0.533944,1000000,200000,2000,0.197583,0.000000,0.027042,533944.0
5,257.784804,998.378096,0.258204,1.254668,0.427512,1000000,200000,2000,0.284875,0.000000,0.034930,427511.0
6,466.378504,994.374994,0.469017,0.122038,0.452122,1000000,200000,2000,0.000333,124.233333,0.087935,452122.0
7,1236.810507,291.798678,4.238575,0.280188,0.536618,1000000,200000,2000,0.066833,81.333333,0.030124,536618.0
8,1432.425694,1120.776878,1.278065,1.291657,0.425042,1000000,200000,2000,0.021125,14.583333,0.388175,425042.0
9,1110.518255,222.505870,4.990962,0.453612,0.650389,1000000,200000,2000,0.344042,0.000000,0.011987,650388.0


In [8]:
# -----------------------------
# 6) Fit surrogate models (dropout + p10)
# -----------------------------
dropout_model, p10_model, feature_names = mod.fit_surrogate_models(rows)
print('Features:', feature_names)


Features: ['HDR_ng', 'sgRNA_ng', 'ratio', 'skew_sigma', 'mapping_rate', 'reads_total', 'cells_transfected', 'library_size']


In [9]:
# -----------------------------
# 7) Option B: surrogate search + Monte Carlo verification
# -----------------------------
df_verified = mod.suggest_experiments_surrogate_verified(
    dropout_model=dropout_model,
    p10_model=p10_model,
    feature_names=feature_names,
    n_candidates=N_CANDIDATES,
    top_k_to_verify=TOP_K_VERIFY,
    target_dropout=TARGET_DROPOUT,
    min_p10_reads=MIN_P10_READS,
    reads_total=READS_TOTAL,
    cells_transfected=CELLS_TRANSFECTED,
    library_size=LIBRARY_SIZE,
    ratio_opt=RATIO_OPT,
    n_reps_verify=N_REPS_VERIFY,
    mapping_range=MAPPING_RANGE_SYNTH,
)

out_csv = os.path.join(OUTDIR, 'suggested_experiments_surrogate_verified.csv')
df_verified.to_csv(out_csv, index=False)
print('Saved verified suggestions CSV:', out_csv)
df_verified.head(20)


Saved verified suggestions CSV: ./suggested_experiments_surrogate_verified.csv


,HDR_ng,sgRNA_ng,ratio,skew_sigma,mapping_rate,reads_total,cells_transfected,library_size,dropout_pred,p10_reads_pred,hdr_rate_pred,mapping_rate_assumed,reads_usable_expected,dropout_sim_mean,p10_reads_sim_mean,dropout_sim_q05,dropout_sim_q95,p10_sim_q05,p10_sim_q95
5,1916.117334,1432.482725,1.337620,0.107526,0.661754,1000000.0,200000.0,2000.0,4.919288e-07,587.891767,0.434941,0.661754,661754.323197,0.0,251.2075,0.0,0.0,247.950,254.050
54,1775.444850,1396.342634,1.271497,0.132577,0.679705,1000000.0,200000.0,2000.0,1.048604e-06,454.054631,0.410742,0.679705,679704.657372,0.0,250.3375,0.0,0.0,247.000,253.050
41,1711.209914,1431.589372,1.195322,0.110564,0.640496,1000000.0,200000.0,2000.0,9.262746e-07,454.816301,0.386706,0.640496,640496.316277,0.0,239.7125,0.0,0.0,237.900,242.050
36,1792.344541,1485.558504,1.206512,0.153254,0.656580,1000000.0,200000.0,2000.0,8.669325e-07,487.044863,0.393906,0.656580,656579.772876,0.0,236.1550,0.0,0.0,232.950,239.905
57,1965.830093,1027.606535,1.913018,0.108437,0.609760,1000000.0,200000.0,2000.0,1.073026e-06,375.825140,0.518702,0.609760,609759.664853,0.0,234.3050,0.0,0.0,232.000,237.000
59,1988.632926,1165.154617,1.706755,0.165253,0.643686,1000000.0,200000.0,2000.0,1.092633e-06,409.961164,0.508517,0.643686,643685.663163,0.0,233.2525,0.0,0.0,228.995,236.000
56,1794.569403,1474.742606,1.216870,0.178991,0.664766,1000000.0,200000.0,2000.0,1.072720e-06,457.215849,0.396899,0.664766,664766.418717,0.0,232.3850,0.0,0.0,228.945,235.905
45,1750.420419,1411.588855,1.240036,0.128524,0.625680,1000000.0,200000.0,2000.0,9.328211e-07,446.616411,0.401012,0.625680,625679.762289,0.0,230.4600,0.0,0.0,228.000,233.045
27,1999.528100,1156.480238,1.728977,0.133271,0.600417,1000000.0,200000.0,2000.0,7.826107e-07,435.692347,0.511507,0.600417,600417.332018,0.0,225.6550,0.0,0.0,222.950,228.000
13,1978.750035,1387.738663,1.425881,0.160672,0.623265,1000000.0,200000.0,2000.0,5.850711e-07,531.410501,0.459326,0.623265,623264.646360,0.0,225.1150,0.0,0.0,221.995,228.050


## What to report from this notebook
- **Predicted HDR rate**: `hdr_rate_pred` (per candidate) and `hdr_rate_mean` (Monte Carlo output)
- **Predicted/assumed mapping rate**: `mapping_rate_assumed` (per candidate) and `mapping_rate_mean` (Monte Carlo output)
- **Usable reads**: `reads_usable_expected` and `reads_usable_mean`
- **Coverage robustness**: `dropout_sim_mean`, `p10_reads_sim_mean` (plus q05/q95 intervals)

The `suggested_experiments_surrogate_verified.csv` file is the main deliverable for “what to try next”.
